# Tarea opcional: punto flotante

## Borrador de soluciones y experimentos en Julia

**Autor:** Mateo Gabriel González Lara. **Curso:** Análisis Numérico.

**Estado: borrador avanzado para revisión y ejecución.** Se incluyen las soluciones matemáticas y código Julia para los siete problemas. Este archivo NO se ha ejecutado en Julia: el entorno de preparación no dispone de ese ejecutable. No se han fabricado salidas ni tiempos. Las comprobaciones independientes en Python están separadas en `validacion_referencia.json` y no sustituyen ejecutar el cuaderno con IJulia.

**Fuente matemática principal:** Kincaid y Cheney, *Numerical Analysis: Mathematics of Scientific Computing*, edición adjunta: §1.3 (ecuaciones en diferencias, pp. 28–36), §2.1 (representación y redondeo, pp. 37–54), §2.2 (errores y pérdida de significancia, pp. 55–63), §2.3 (estabilidad, pp. 64–71). Las páginas son las impresas. No se atribuyen al libro fórmulas de los enunciados que proceden de otras fuentes.

El problema 2 menciona Python, pero se usa Julia siguiendo la instrucción general. El problema 5 contiene una inconsistencia de índices que se conserva y se explica antes de adoptar una interpretación.


## Preparación y convenciones

Para ejecutar, abrir con un kernel Julia/IJulia. Se usa `BenchmarkTools` solo en el problema 7. Si falta, ejecutar una vez en Julia `using Pkg; Pkg.add("BenchmarkTools")`. La instalación no se realiza automáticamente desde este cuaderno.

Se distinguen el resultado exacto, el valor calculado y una referencia de mayor precisión. Para una referencia no nula $r$ se usan
$$E_a=|\widehat y-r|,\qquad E_r=\frac{|\widehat y-r|}{|r|}.$$

La representación `Float64` tiene 53 bits de precisión; con redondeo al más cercano su unidad de redondeo es $u=2^{-53}$, mientras que `eps(Float64) = 2^{-52}` es la separación entre 1 y su sucesor. No se confunden estas constantes con las de Marc-32. El modelo de error relativo de §2.1 se usa cuando las operaciones permanecen en el rango normal y no hay desbordamiento ni subdesbordamiento.


In [ ]:
using Printf
using Statistics
using InteractiveUtils

println("Versión Julia: ", VERSION)
println("Bytes por Float64: ", sizeof(Float64))
println("eps(Float64): ", eps(Float64))
println("Unidad de redondeo: ", eps(Float64) / 2)


## Problema 1. Memoria

**Enunciado.** ¿Cuánta memoria requieren 100 000 números de doble precisión y una matriz de dimensión $100\,000\times100\,000$?

Cada `Float64` ocupa $64/8=8$ bytes. Por tanto,
$$M_v=100\,000(8)=800\,000\ \text{bytes}=0.8\ \text{MB}=781.25\ \text{KiB},$$
$$M_A=(100\,000)^2(8)=80\,000\,000\,000\ \text{bytes}=80\ \text{GB}\approx74.505806\ \text{GiB}.$$

Se usan $1\text{ MB}=10^6$ bytes, $1\text{ GB}=10^9$ bytes y $1\text{ GiB}=2^{30}$ bytes. Se cuenta el contenido numérico de un arreglo denso, sin metadatos del contenedor. No se construye la matriz: el producto de dimensiones basta. Se verifica el tamaño de los datos con un vector pequeño.


In [ ]:
function memoria_float64(N::Integer)
    N > 0 || throw(ArgumentError("N debe ser positivo"))
    b = sizeof(Float64)
    return (vector=big(N)*b, matriz=big(N)^2*b)
end

let M = memoria_float64(100_000)
    println("Vector: ", M.vector, " bytes; ", Float64(M.vector)/10^6, " MB")
    println("Matriz: ", M.matriz, " bytes; ", Float64(M.matriz)/10^9, " GB")
    println("Matriz: ", Float64(M.matriz)/2^30, " GiB")
    v = zeros(Float64, 100)
    @assert sizeof(v) == 100 * sizeof(Float64)
    @assert M.vector == 800_000
    @assert M.matriz == 80_000_000_000
end


## Problema 2. Sucesión por tramos

**Enunciado.** Con $x_0=1/10$ y, después, con $x_0=1/8$, calcular analíticamente y mediante código $x_1,\ldots,x_{50}$, donde
$$T(x)=\begin{cases}2x,&0\le x\le1/2,\\2x-1,&1/2<x\le1.\end{cases}$$

**Solución exacta para $x_0=1/10$.** Los primeros valores son
$$x_1=\tfrac15,\quad x_2=\tfrac25,\quad x_3=\tfrac45,\quad x_4=\tfrac35,\quad x_5=\tfrac15.$$
Como $T$ es una función, la repetición de $x_1$ obliga a repetir los valores siguientes. Así, para todo entero $q\ge0$,
$$x_{4q+1}=\tfrac15,\quad x_{4q+2}=\tfrac25,\quad x_{4q+3}=\tfrac45,\quad x_{4q+4}=\tfrac35.$$
Esto determina todos los valores pedidos y da $x_{50}=2/5$.

**Representación inicial y propagación.** Un racional reducido tiene expansión binaria finita si y solo si su denominador es una potencia de 2; esto se obtiene al escribir una expansión binaria finita sobre un denominador $2^p$, y recíprocamente. Por tanto, $1/10$ no es representable exactamente y $1/8$ sí lo es. En `Float64`, con redondeo al más cercano,
$$\widehat x_0-\tfrac1{10}=\frac1{5\cdot2^{55}}.$$
Mientras ambos recorridos sigan la misma rama, $\widehat x_{n+1}-x_{n+1}=2(\widehat x_n-x_n)$, si la operación es exacta. Para los primeros 50 pasos de este ejemplo las operaciones con el dato binario almacenado son exactas y las ramas coinciden, lo cual también se verifica con racionales en el código. Por tanto,
$$\widehat x_{50}-x_{50}=\frac{2^{50}}{5\cdot2^{55}}=\frac1{160},\qquad \widehat x_{50}=0.40625.$$
El error observado hasta ese punto se debe a la amplificación del error inicial; no hace falta atribuirlo a nuevo redondeo en cada paso. No se extiende la igualdad de errores sin comprobar las ramas, porque $T$ es discontinua en $1/2$.

**Solución para $x_0=1/8$.** Se obtiene exactamente
$$x_1=\tfrac14,\quad x_2=\tfrac12,\quad x_n=1\quad(n\ge3).$$
La rama del enunciado incluye $1/2$ en el primer caso. Sustituirla por el mapa usual módulo 1 cambiaría la respuesta. Todos estos valores son representables exactamente en binario y las operaciones indicadas conservan su exactitud.

**Fundamento:** representación binaria de §2.1 y propagación de perturbaciones de §2.3.


In [ ]:
function orbita(x0, N::Integer)
    N >= 0 || throw(ArgumentError("N debe ser no negativo"))
    zero(x0) <= x0 <= one(x0) || throw(ArgumentError("x0 fuera de [0,1]"))
    xs = Vector{typeof(x0)}(undef, N+1)
    xs[1] = x0
    for n in 1:N
        x = xs[n]
        xs[n+1] = 2*x <= one(x) ? 2*x : 2*x-one(x)
    end
    return xs
end

function tabla_orbita(q0::Rational, N=50)
    exactos = orbita(q0, N)
    flotantes = orbita(Float64(q0), N)
    println(" n      exacto                Float64             error absoluto")
    setprecision(BigFloat, 256) do
        for n in 1:N
            e = abs(BigFloat(flotantes[n+1])-BigFloat(exactos[n+1]))
            @printf("%2d  %10s  % .17e  %.5e\n", n, string(exactos[n+1]), flotantes[n+1], Float64(e))
        end
    end
    return (exactos=exactos, flotantes=flotantes)
end

orbitas10 = tabla_orbita(big(1)//big(10))
orbitas8 = tabla_orbita(big(1)//big(8))

let q10 = big(1)//big(10), qhat = Rational{BigInt}(0.1)
    @assert qhat-q10 == big(1)//(big(5)*big(2)^55)
    exactos_dato_almacenado = orbita(qhat, 50)
    @assert all(Rational{BigInt}(orbitas10.flotantes[j]) == exactos_dato_almacenado[j] for j in 1:51)
    @assert all((2*orbitas10.exactos[j] <= 1) == (2*exactos_dato_almacenado[j] <= 1) for j in 1:50)
    @assert orbitas10.exactos[51] == big(2)//big(5)
    @assert orbitas10.flotantes[51] == 0.40625
    @assert all(Rational{BigInt}(orbitas8.flotantes[j]) == orbitas8.exactos[j] for j in 1:51)
end


## Problema 3. Softmax sin desbordamiento por exceso

**Enunciado.** Implementar de forma estabilizada
$$SF_i(x)=\frac{e^{x_i}}{\sum_{j=1}^n e^{x_j}}.$$

**Dominio del programa:** vector no vacío de números finitos en punto flotante. Sea $a=\max_jx_j$. Dividir numerador y denominador por $e^a$ da
$$SF_i(x)=\frac{e^{x_i-a}}{\sum_{j=1}^n e^{x_j-a}}.$$
Como $x_i-a\le0$, los exponenciales están entre 0 y 1, y algún término es 1. Se evita así el desbordamiento por exceso de los exponenciales y el denominador no es cero. En aritmética exacta todas las componentes son positivas y suman 1; en punto flotante la suma es aproximada y componentes extremadamente pequeñas pueden subdesbordar a cero. Esto no garantiza exactitud relativa de probabilidades arbitrariamente pequeñas. En un vector tan grande que su longitud superara el rango del acumulador, también habría que estudiar la suma; no ocurre en los experimentos propuestos.

La identidad se demuestra aquí por álgebra; §2.1 explica los límites de representación y §2.2–2.3 fundamentan comparar formulaciones equivalentes por su comportamiento numérico. No se afirma que el libro presente softmax.


In [ ]:
softmax_directa(x::AbstractVector{<:AbstractFloat}) = exp.(x) ./ sum(exp.(x))

function softmax_estable(x::AbstractVector{T}) where {T<:AbstractFloat}
    isempty(x) && throw(ArgumentError("El vector no puede estar vacío"))
    all(isfinite, x) || throw(ArgumentError("Se requieren entradas finitas"))
    a = maximum(x)
    y = exp.(x .- a)
    return y ./ sum(y)
end

for x in ([1.0,2.0,3.0], [1000.0,1001.0,1002.0], [-1000.0,-1001.0,-1002.0])
    p = softmax_estable(x)
    println("x = ", x)
    println("Directa: ", softmax_directa(x))
    println("Estable: ", p, "; suma = ", sum(p))
    @assert all(isfinite, p)
    @assert all(t -> t >= 0, p)
    @assert isapprox(sum(p), 1.0; atol=10*eps(Float64), rtol=0)
end

let p = softmax_estable([1000.0,1001.0,1002.0]), q = softmax_estable([0.0,1.0,2.0])
    @assert isapprox(p, q; atol=10*eps(Float64), rtol=0)
end


## Problema 4. Cancelación en dos expresiones equivalentes

**Enunciado.** Comparar, para $x=8^{-1},8^{-2},\ldots$,
$$f(x)=\sqrt{1+x^2}-1,\qquad g(x)=\frac{x^2}{\sqrt{1+x^2}+1}.$$

Multiplicar por el conjugado demuestra $f(x)=g(x)$ para todo real $x$, porque $\sqrt{1+x^2}+1>0$. Sin embargo, cuando $x$ es pequeño, la primera fórmula resta números casi iguales. Puesto que $\sqrt{1+x^2}-1\sim x^2/2$, un error absoluto de orden $u$ en la raíz puede producir un error relativo de orden $u/x^2$. La fórmula racionalizada evita esa resta.

En `Float64`, para $k=9$, $x=8^{-9}=2^{-27}$ y $x^2=2^{-54}$; redondear $1+2^{-54}$ al más cercano produce 1. Así, la primera fórmula devuelve cero, aunque el valor matemático es positivo. La segunda conserva un resultado positivo cercano a $2^{-55}$. La referencia de 256 bits se calcula también con la fórmula racionalizada, sin convertir primero una referencia calculada en `Float64`.

Esta reformulación coincide con el ejemplo 1 de §2.2, p. 56, y el experimento es el problema computacional 1 de esa sección, p. 62. La aplicación cuantitativa del teorema de pérdida de precisión, §2.2, p. 57, debe distinguir la resta de dos números de máquina del error ya introducido al evaluar la raíz.

El análisis se limita a $1\le k\le30$; para valores muchísimo menores también sería necesario estudiar el subdesbordamiento de $x^2$. No se afirma que la fórmula racionalizada elimine todos los errores para cualquier entrada.


In [ ]:
f_cancelacion(x) = sqrt(one(x)+x*x)-one(x)
g_racionalizada(x) = (x*x)/(sqrt(one(x)+x*x)+one(x))

datos_cancelacion = setprecision(BigFloat, 256) do
    filas = NamedTuple[]
    for k in 1:30
        x = ldexp(1.0, -3*k)
        referencia = g_racionalizada(BigFloat(x))
        fx, gx = f_cancelacion(x), g_racionalizada(x)
        ef = abs(BigFloat(fx)-referencia)/abs(referencia)
        eg = abs(BigFloat(gx)-referencia)/abs(referencia)
        push!(filas, (k=k, x=x, f=fx, g=gx, error_f=Float64(ef), error_g=Float64(eg)))
    end
    filas
end

println(" k         x                f(x)              g(x)          Er(f)       Er(g)")
for r in datos_cancelacion
    @printf("%2d  %.6e  %.10e  %.10e  %.3e  %.3e\n", r.k,r.x,r.f,r.g,r.error_f,r.error_g)
end
@assert datos_cancelacion[9].f == 0.0
@assert datos_cancelacion[9].g > 0.0


## Problema 5. Recurrencia de Viète: indexación y estabilidad

**Enunciado original, conservado sin cambios:**
$$z_1=2,\qquad z_{n+1}=2^{n-1/2}\sqrt{1-\sqrt{1-4^{1-n}z_n^2}},\qquad n=2,3,\ldots.$$

**Inconsistencia.** Con $z_1$ y una recurrencia que empieza en $n=2$, falta $z_2$. Empezar simplemente en $n=1$ tampoco resuelve el problema: el radicando interior sería $1-2^2=-3$. No se cambia el enunciado en silencio.

**Interpretación propuesta, pendiente de contrastar con el docente o la fuente original.** Usar la fórmula dada desde $n=2$ con $z_2=2$. Esto equivale a reemplazar el dato inicial $z_1=2$ por $z_2=2$. Se justifica porque conduce a
$$z_n=2^{n-1}\sin\!\left(\frac{\pi}{2^{n-1}}\right),\quad n\ge2.$$
En efecto, con $\theta=\pi/2^{n-1}\in(0,\pi/2]$, se tiene $4^{1-n}z_n^2=\sin^2\theta$, $\sqrt{1-\sin^2\theta}=\cos\theta$ y $1-\cos\theta=2\sin^2(\theta/2)$, de donde se obtiene la fórmula para $z_{n+1}$. Por $\sin t/t\to1$, el límite es $\pi$.

Para una implementación sin huecos definimos $w_k=z_{k+1}$; entonces
$$w_1=2,\qquad w_{k+1}=2^{k+1/2}\sqrt{1-\sqrt{1-4^{-k}w_k^2}},\quad k\ge1,$$
$$w_k=2^k\sin\!\left(\frac{\pi}{2^k}\right)\longrightarrow\pi.$$

**Reformulación estable.** Con $a_k=4^{-k}w_k^2$, racionalizar da
$$1-\sqrt{1-a_k}=\frac{a_k}{1+\sqrt{1-a_k}},$$
y, como $w_k>0$,
$$\boxed{w_{k+1}=w_k\sqrt{\frac{2}{1+\sqrt{1-a_k}}}.}$$
La fórmula original resta dos cantidades cada vez más próximas; la racionalizada evita esa cancelación. La resta interior $1-a_k$ merece cuidado en el primer paso ($a_1=1$ exactamente), pero después $a_k$ decrece y esa resta no presenta la misma cancelación persistente. La versión estable también termina estancándose por precisión finita; no alcanza precisión ilimitada.

Por Taylor, $\sin t=t-t^3/6+O(t^5)$, por lo que
$$\pi-w_k=\frac{\pi^3}{6\,4^k}+O(16^{-k}).$$
En aritmética exacta el error decrece aproximadamente por un factor 4. En punto flotante el redondeo acaba dominando. Se aplican Taylor (§1.1) y la pérdida de significancia y estabilidad de §2.2–2.3; la corrección de índices es una deducción explícita, no una errata confirmada por el libro adjunto.


In [ ]:
function viete(N::Integer, ::Type{T}=Float64; estable=false) where {T<:AbstractFloat}
    N >= 1 || throw(ArgumentError("N debe ser al menos 1"))
    w = Vector{T}(undef, N)
    w[1] = T(2)
    for k in 1:N-1
        a = ldexp(w[k], -k)^2
        r = one(T)-a
        r >= zero(T) || throw(DomainError(r, "Radicando negativo; revisar índice o redondeo"))
        interior = sqrt(r)
        if estable
            w[k+1] = w[k]*sqrt(T(2)/(one(T)+interior))
        else
            # Equivale a 2^(k+1/2)*sqrt(1-sqrt(1-a)).
            # Esta elección del orden de operaciones debe registrarse al comparar runtimes.
            w[k+1] = ldexp(sqrt(T(2)*(one(T)-interior)), k)
        end
    end
    return w
end

viete_directa64 = viete(40)
viete_estable64 = viete(40; estable=true)
datos_viete = setprecision(BigFloat, 256) do
    pi_ref = BigFloat(pi)
    [(k=k, directa=viete_directa64[k], estable=viete_estable64[k],
      error_directa=Float64(abs(BigFloat(viete_directa64[k])-pi_ref)),
      error_estable=Float64(abs(BigFloat(viete_estable64[k])-pi_ref))) for k in 1:40]
end

println(" k       directa             estable           Ea(directa)   Ea(estable)")
for r in datos_viete
    @printf("%2d  %.16e  %.16e  %.4e  %.4e\n", r.k,r.directa,r.estable,r.error_directa,r.error_estable)
end
@assert isapprox(viete_estable64[end], Float64(pi); atol=20*eps(Float64), rtol=0)

setprecision(BigFloat, 256) do
    alta = viete(80, BigFloat; estable=true)
    println("Error a 256 bits, k=80: ", abs(alta[end]-BigFloat(pi)))
end


## Problema 6. Recurrencia de Stewart

**Enunciado.** Implementar $x_1=1/3$, $x_2=1/12$ y
$$x_{k+1}=2.25x_k-0.5x_{k-1},\qquad k\ge2,$$
y explicar los resultados.

Es una ecuación lineal en diferencias con coeficientes constantes (§1.3). Su polinomio característico es
$$r^2-\tfrac94r+\tfrac12=(r-2)(r-\tfrac14).$$
Las raíces son distintas y no nulas; por la representación de soluciones de §1.3,
$$x_k=A\left(\tfrac14\right)^{k-1}+B\,2^{k-1}.$$
Los datos dan $A+B=1/3$ y $A/4+2B=1/12$, de donde $B=0$ y $A=1/3$. Así,
$$\boxed{x_k=\frac1{3\,4^{k-1}}\longrightarrow0.}$$

**Inestabilidad.** Si los primeros datos cambian en $e_1,e_2$ y después se itera exactamente, el coeficiente de la componente creciente del error es
$$B_e=\frac{4e_2-e_1}{7}.$$
Una componente no nula proporcional a $2^{k-1}$ acaba dominando la solución deseada proporcional a $4^{1-k}$; su importancia relativa crece como $8^{k-1}$. Además, si cada paso introduce una perturbación local $\rho_k$, el error satisface
$$e_{k+1}=\tfrac94e_k-\tfrac12e_{k-1}+\rho_k,$$
lo que permite volver a introducir la componente creciente durante el cálculo. Este es el mecanismo de §2.3: perturbaciones pequeñas amplificadas por la recurrencia.

**Matiz importante.** En `Float64`, los valores iniciales almacenados cumplen exactamente $\widehat x_2=\widehat x_1/4$; sus errores iniciales están correlacionados y cumplen $4e_2-e_1=0$. Por tanto, no es correcto atribuir en este ejemplo toda la componente creciente únicamente al redondeo inicial. El redondeo de las operaciones posteriores la introduce.

Para esta solución concreta, la recurrencia de primer orden $x_{k+1}=x_k/4$, partiendo de $1/3$, evita ese modo creciente y escala exactamente por una potencia de 2 mientras no se encuentre subdesbordamiento. Esta alternativa usa la solución analítica y no reemplaza el análisis de la recurrencia pedida.


In [ ]:
function stewart(N::Integer, ::Type{T}=Float64) where {T<:AbstractFloat}
    N >= 2 || throw(ArgumentError("N debe ser al menos 2"))
    x = Vector{T}(undef, N)
    x[1], x[2] = one(T)/T(3), one(T)/T(12)
    for k in 2:N-1
        x[k+1] = (T(9)/T(4))*x[k] - (one(T)/T(2))*x[k-1]
    end
    return x
end

function stewart_alternativa(N::Integer)
    x = Vector{Float64}(undef, N)
    x[1] = 1.0/3.0
    for k in 1:N-1
        x[k+1] = x[k]/4.0
    end
    return x
end

stewart64 = stewart(80)
stewart_alt64 = stewart_alternativa(80)
datos_stewart = setprecision(BigFloat, 256) do
    [(k=k, calculado=stewart64[k], alternativo=stewart_alt64[k],
      referencia=inv(BigFloat(3)*BigFloat(4)^(k-1)),
      error_relativo=Float64(abs(BigFloat(stewart64[k])*BigFloat(3)*BigFloat(4)^(k-1)-1))) for k in 1:80]
end

println(" k       recurrencia          referencia          alternativa      Er(recurrencia)")
for k in (1,2,3,10,20,30,40,50,80)
    r = datos_stewart[k]
    @printf("%2d  %.12e  %.12e  %.12e  %.4e\n",r.k,r.calculado,Float64(r.referencia),r.alternativo,r.error_relativo)
end
@assert stewart64[2] == stewart64[1]/4

setprecision(BigFloat, 256) do
    alta = stewart(80, BigFloat)
    println("Error relativo a 256 bits, k=80: ", abs(alta[end]*BigFloat(3)*BigFloat(4)^79-1))
end


## Problema 7. `@time`, `@btime` y `@benchmark`

**Enunciado.** Investigar estas utilidades y mostrar ejemplos con operaciones de punto flotante.

| Utilidad | Qué hace | Qué observar |
|---|---|---|
| `@time` (Julia Base) | Ejecuta la expresión una vez, muestra tiempo y asignaciones y devuelve su resultado | La primera llamada puede incluir compilación; comparar llamadas sucesivas |
| `@btime` (BenchmarkTools) | Realiza mediciones repetidas, imprime una estimación basada en el menor tiempo y las asignaciones, y devuelve el resultado de la expresión | Interpolar entradas externas con `$`; no interpretar el mínimo como mediana |
| `@benchmark` (BenchmarkTools) | Devuelve un `Trial` con las muestras de tiempo y memoria | Comparar mínimo y mediana, además de la dispersión |

Se mide una función que recorre un vector ya construido; ni su construcción ni la impresión se incluyen dentro de la operación medida. Se hace una llamada de calentamiento antes de BenchmarkTools. El tiempo no prueba estabilidad: la comparación de precisión debe mantenerse separada. No se usan `@fastmath` ni cambios del orden de operaciones para acelerar los ejemplos, porque alterarían los experimentos de redondeo.

**Fuentes técnicas consultadas:** [Julia, Performance Tips](https://docs.julialang.org/en/v1/manual/performance-tips/) y [BenchmarkTools, Manual](https://juliaci.github.io/BenchmarkTools.jl/stable/manual/), consulta 5 de noviembre de 2026. Estas fuentes describen las herramientas; el fundamento matemático sigue siendo Kincaid y Cheney.

**Pendiente:** ejecutar esta sección y registrar tiempos y entorno reales. No hay tiempos anticipados.


In [ ]:
using BenchmarkTools

function suma_cuadrados(v::AbstractVector{T}) where {T<:AbstractFloat}
    s = zero(T)
    for x in v
        s += x*x
    end
    return s
end

vector_tiempos = collect(range(0.0, 1.0; length=100_000))

# La primera llamada puede incluir compilación. La segunda usa el mismo tipo.
resultado_primera = @time suma_cuadrados(vector_tiempos)
resultado_segunda = @time suma_cuadrados(vector_tiempos)
@assert resultado_primera == resultado_segunda

suma_cuadrados(vector_tiempos) # calentamiento explícito
@btime suma_cuadrados($vector_tiempos) seconds=1
ensayo = @benchmark suma_cuadrados($vector_tiempos) seconds=1
display(ensayo)
println("Mínimo (ns): ", minimum(ensayo).time)
println("Mediana (ns): ", median(ensayo).time)
println("Memoria estimada (bytes): ", minimum(ensayo).memory)
versioninfo()


## Interpretación conjunta y pendientes de cierre

1. Representar un dato y ejecutar operaciones son fuentes distintas de error: el problema 2 permite aislar el error inicial.
2. Dos fórmulas equivalentes pueden tener comportamientos numéricos muy diferentes: problemas 3, 4 y 5.
3. Una recurrencia que define una solución convergente puede amplificar errores hasta dominarla: problema 6.
4. El costo de ejecución requiere mediciones reales; no se deduce de una sola llamada con compilación.

**Antes de entregar:** ejecutar todas las celdas en un kernel Julia nuevo; corregir problemas de compatibilidad; guardar salidas auténticas; agregar gráficas de errores a partir de `datos_cancelacion`, `datos_viete` y `datos_stewart`; verificar la notación con lo visto en clase; confirmar o discutir la interpretación de índices del problema 5; revisar la reflexión del autor y la declaración de IA. El código es un borrador sin ejecución Julia, aunque los mecanismos centrales se contrastaron independientemente en Python.

**Reflexión del autor: pendiente de redacción personal.** Explicar qué resultado sorprendió, qué comprobación se realizó personalmente y qué modificación del código fue necesaria. No presentar esta indicación como experiencia ya realizada.

**Declaración de apoyo de IA, para revisar:** Se utilizó ChatGPT para organizar el cuaderno, elaborar un borrador de explicaciones y código, detectar la inconsistencia de índices del problema 5 y realizar comprobaciones independientes. La ejecución Julia, la revisión contra el material de clase y la validación final todavía están pendientes. Actualizar esta declaración al completar esas tareas.

## Referencias

- Kincaid, D. y Cheney, W. (2002). *Numerical Analysis: Mathematics of Scientific Computing*. Edición proporcionada: §§1.1, 1.3 y 2.1–2.3.
- Cuaderno de enunciados *ProblemasPuntoFlotanteJulia.ipynb*, elaborado por Juan Galvis. Se conserva por separado en el paquete de continuación.
- Julia Documentation. *Performance Tips*. https://docs.julialang.org/en/v1/manual/performance-tips/
- JuliaCI. *BenchmarkTools.jl Manual*. https://juliaci.github.io/BenchmarkTools.jl/stable/manual/

Los nombres Sullivan, Goodfellow, Stoer, Quarteroni–Saleri–Gervasio y Stewart se conservan como atribuciones del cuaderno original; no se afirma haber consultado sus obras directamente.
